# EPMEI Stage 2 Training with A4 (Marker-Aligned GCN) and A5 (Lexicon Sentiment Prior)
Aspect Sentiment Triplet Extraction (ASTE) on 14lap, 14res, 15res, 16res.
Fully resumable across multiple Kaggle sessions (handles the 12-hour GPU limit).

In [ ]:
import os
import sys
import subprocess
import json
import shutil
import glob
import time
import re
import statistics
import torch

SESSION_START_TIME = time.time()
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


## Workspace & Dependencies Setup

In [ ]:
ROOT_DIR = "/kaggle/working"
REPO_DIR = os.path.join(ROOT_DIR, "EPMEI")
DATA_DIR = os.path.join(REPO_DIR, "data/ASTE-Data-V2-EMNLP2020_pro")
OUTPUT_DIR = os.path.join(ROOT_DIR, "EPMEI_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

for cand in glob.glob("/kaggle/input/*/EPMEI_outputs") + glob.glob("/kaggle/input/*/sc"):
    base_src = cand if cand.endswith("EPMEI_outputs") else os.path.dirname(cand)
    if os.path.exists(base_src):
        print("Found previous run outputs to resume from:", base_src)
        for item in os.listdir(base_src):
            src_item = os.path.join(base_src, item)
            dst_item = os.path.join(OUTPUT_DIR, item)
            if not os.path.exists(dst_item):
                if os.path.isdir(src_item):
                    shutil.copytree(src_item, dst_item)
                else:
                    shutil.copy2(src_item, dst_item)
        print("Resume sync complete.")
        break

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "boto3", "filelock", "regex", "sacremoses",
                "sentencepiece", "six", "tokenizers", "tensorboardX", "wandb", "nltk"], check=True)


## Setup Repository and Editable Transformers

In [ ]:
if not os.path.exists(os.path.join(REPO_DIR, "run_sc.py")):
    if os.path.exists("/kaggle/input/epmei-codebase"):
        shutil.copytree("/kaggle/input/epmei-codebase", REPO_DIR)
    else:
        subprocess.run(["git", "clone", "https://github.com/Ishant89op/EPMEI.git", REPO_DIR], check=True)

for fname in ["run_ner.py", "run_sc.py"]:
    fpath = os.path.join(REPO_DIR, fname)
    if os.path.exists(fpath):
        s = open(fpath).read()
        s = s.replace("from allennlp.modules.token_embedders.embedding import Embedding",
                      "from torch.nn import Embedding")
        open(fpath, "w").write(s)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
               cwd=os.path.join(REPO_DIR, "transformers"), check=True)


## Setup Lexicon (Hu & Liu Opinion Lexicon)

In [ ]:
LEXICON_DIR = os.path.join(REPO_DIR, "lexicon")
os.makedirs(LEXICON_DIR, exist_ok=True)
pos_file = os.path.join(LEXICON_DIR, "positive-words.txt")
neg_file = os.path.join(LEXICON_DIR, "negative-words.txt")

if not (os.path.exists(pos_file) and os.path.exists(neg_file)):
    os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
    import nltk
    nltk.download("opinion_lexicon")
    from nltk.corpus import opinion_lexicon
    with open(pos_file, "w", encoding="latin-1") as f:
        for w in opinion_lexicon.positive():
            print(w, file=f)
    with open(neg_file, "w", encoding="latin-1") as f:
        for w in opinion_lexicon.negative():
            print(w, file=f)

print("Lexicon ready at:", LEXICON_DIR)


## Verify lexicon.py module

In [ ]:
lex_file = os.path.join(REPO_DIR, "lexicon.py")
assert os.path.exists(lex_file), f"{lex_file} not found. Please pull the latest repo."
print("lexicon.py found and verified at:", lex_file)


## Helper Functions & Execution Runner

In [ ]:
TOKENIZER_FILES = ["vocab.txt", "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json"]

def best_test_metrics(results_file):
    if not os.path.exists(results_file):
        return None
    try:
        r = json.load(open(results_file))
        steps = {m.group(2) for k in r for m in [re.match(r"^(f1|p|r|f1_with_ner|ner_f1|aope_f1)_(\d*)$", k)] if m}
        if not steps:
            return None
        step = max(steps, key=lambda s: int(s) if s.isdigit() else -1)
        out = {k: r[f"{k}_{step}"] for k in ["f1", "p", "r", "f1_with_ner", "ner_f1"] if f"{k}_{step}" in r}
        out["dev_best_f1"] = r.get("dev_best_f1")
        out["step"] = step
        return out
    except Exception:
        return None

def is_sc_done(run_dir):
    res_file = os.path.join(run_dir, "results.json")
    m = best_test_metrics(res_file)
    return m is not None and "f1" in m

def pack_model(run_dir):
    cps = glob.glob(f"{run_dir}/checkpoint-*")
    if not cps:
        return None
    best = max(cps, key=lambda p: int(p.rsplit("-", 1)[-1]))
    model_dir = f"{run_dir}/model"
    shutil.rmtree(model_dir, ignore_errors=True)
    os.rename(best, model_dir)
    for name in TOKENIZER_FILES:
        if os.path.exists(f"{run_dir}/{name}"):
            shutil.copy2(f"{run_dir}/{name}", f"{model_dir}/{name}")
    for cp in glob.glob(f"{run_dir}/checkpoint-*"):
        shutil.rmtree(cp, ignore_errors=True)
    return model_dir

def run_sc(dataset, seed, align_markers=True, use_lexicon=True, ner_pred_file=None):
    out_dir = f"{OUTPUT_DIR}/sc/{dataset}/seed{seed}"
    if is_sc_done(out_dir):
        print(f"Dataset {dataset} seed {seed} already completed. Loading existing results.")
        return best_test_metrics(f"{out_dir}/results.json")
    os.makedirs(out_dir, exist_ok=True)
    if ner_pred_file is None:
        ner_pred_file = f"{REPO_DIR}/data/ner_predictions/{dataset}/ent_pred_test_pro.json"
        if not os.path.exists(ner_pred_file):
            alt = f"/kaggle/input/epmei-ner-results/{dataset}/ent_pred_test_pro.json"
            if os.path.exists(alt):
                ner_pred_file = alt
            else:
                os.makedirs(os.path.dirname(ner_pred_file), exist_ok=True)
                url = f"https://raw.githubusercontent.com/Ishant89op/EPMEI/main/data/ner_predictions/{dataset}/ent_pred_test_pro.json"
                print(f"Downloading {dataset} NER predictions from GitHub...")
                import urllib.request
                urllib.request.urlretrieve(url, ner_pred_file)
    cmd = [
        sys.executable, "-u", "run_sc.py",
        "--model_type", "bertsub",
        "--model_name_or_path", "bert-base-uncased",
        "--do_lower_case",
        "--data_dir", f"{DATA_DIR}/{dataset}/",
        "--learning_rate", "4e-5",
        "--num_train_epochs", "10",
        "--per_gpu_train_batch_size", "8",
        "--per_gpu_eval_batch_size", "16",
        "--gradient_accumulation_steps", "1",
        "--max_seq_length", "256",
        "--max_pair_length", "16",
        "--save_steps", "50",
        "--save_total_limit", "1",
        "--do_train", "--do_eval",
        "--evaluate_during_training",
        "--eval_all_checkpoints",
        "--eval_logsoftmax",
        "--seed", str(seed),
        "--test_file", ner_pred_file,
        "--use_ner_results",
        "--use_typemarker",
        "--gcn", "--n_gcn", "2",
        "--output_dir", out_dir,
        "--overwrite_output_dir",
        "--focalloss"
    ]
    if align_markers:
        cmd.append("--align_markers")
    if use_lexicon:
        cmd.extend(["--use_lexicon", "--lexicon_dir", LEXICON_DIR, "--lex_emb_dim", "32"])
    log_path = f"{out_dir}/train.log"
    with open(log_path, "w") as f:
        p = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            f.write(line)
            if re.search(r"Result|Epoch|best|F1|f1|saving|Saving|Error|Traceback", line):
                print(line.rstrip())
        p.wait()
        if p.returncode != 0:
            raise RuntimeError(f"Training failed with code {p.returncode}. Check {log_path}")
    pack_model(out_dir)
    return best_test_metrics(f"{out_dir}/results.json")


## Smoke Test (14lap, Seed 43, A4 + A5)

In [ ]:
smoke_metrics = run_sc("14lap", 43, align_markers=True, use_lexicon=True)
print("Smoke test results (14lap, Seed 43):")
print(json.dumps(smoke_metrics, indent=2))


## Full Benchmark Sweep (4 Datasets x 5 Seeds)

In [ ]:
DATASETS = ["14lap", "14res", "15res", "16res"]
SEEDS = [41, 42, 43, 44, 45]
BASELINE_F1 = {"14lap": 65.25, "14res": 80.52, "15res": 71.44, "16res": 78.88}
PAPER_F1 = {"14lap": 69.64, "14res": 80.98, "15res": 73.01, "16res": 77.41}
MAX_HOURS = 11.0
all_results = {}

for ds in DATASETS:
    all_results[ds] = []
    print(f"=== Starting dataset: {ds} ===")
    for seed in SEEDS:
        elapsed = (time.time() - SESSION_START_TIME) / 3600
        if elapsed >= MAX_HOURS:
            print(f"Reached session time limit ({elapsed:.1f}h / {MAX_HOURS}h). Gracefully saving progress...")
            break
        print(f"Running {ds} seed {seed}...")
        m = run_sc(ds, seed, align_markers=True, use_lexicon=True)
        if m and "f1" in m:
            score = m["f1"] * 100 if m["f1"] <= 1.0 else m["f1"]
            all_results[ds].append(score)
            print(f"Done {ds} seed {seed}: F1 = {score:.2f}")
    if (time.time() - SESSION_START_TIME) / 3600 >= MAX_HOURS:
        break


## Summary and Comparison Table

In [ ]:
summary_rows = []
for ds in DATASETS:
    f1s = all_results.get(ds, [])
    if f1s:
        mean_f1 = statistics.mean(f1s)
        std_f1 = statistics.stdev(f1s) if len(f1s) > 1 else 0.0
        delta_base = mean_f1 - BASELINE_F1[ds]
        delta_paper = mean_f1 - PAPER_F1[ds]
        summary_rows.append({
            "dataset": ds,
            "our_f1_a4_a5": round(mean_f1, 2),
            "std": round(std_f1, 2),
            "completed_seeds": len(f1s),
            "baseline_f1": BASELINE_F1[ds],
            "delta_vs_baseline": round(delta_base, 2),
            "paper_f1": PAPER_F1[ds],
            "delta_vs_paper": round(delta_paper, 2)
        })

print(f"{'Dataset':<8} | {'Done':<5} | {'A4+A5 F1':<10} | {'Baseline':<10} | {'Delta (Base)':<12} | {'Paper F1':<10} | {'Delta (Paper)':<12}")
print("-" * 84)
for r in summary_rows:
    print(f"{r['dataset']:<8} | {r['completed_seeds']:<5} | {r['our_f1_a4_a5']:<10.2f} | {r['baseline_f1']:<10.2f} | {r['delta_vs_baseline']:<+12.2f} | {r['paper_f1']:<10.2f} | {r['delta_vs_paper']:<+12.2f}")

with open(f"{OUTPUT_DIR}/results_comparison.csv", "w") as f:
    print("dataset,completed_seeds,our_f1_a4_a5,std,baseline_f1,delta_vs_baseline,paper_f1,delta_vs_paper", file=f)
    for r in summary_rows:
        print(f"{r['dataset']},{r['completed_seeds']},{r['our_f1_a4_a5']},{r['std']},{r['baseline_f1']},{r['delta_vs_baseline']},{r['paper_f1']},{r['delta_vs_paper']}", file=f)
